
---

## 📘 1 — Project Introduction

# 🎤 AI-Based Speech Emotion Recognition System (RAVDESS)

This project provides:

* 🎧 Audio (.wav) Upload Support
* 🤖 Deep Learning based Emotion Detection
* 🧠 CNN Model with MFCC Features
* 📊 Multi-Class Emotion Classification
* 🌍 Public Deployment using Flask + ngrok

### ✅ Supported Emotion Classes:

* Neutral
* Calm
* Happy
* Sad
* Angry
* Fearful
* Disgust
* Surprised

This notebook performs:

1. Dataset Extraction from Google Drive
2. Audio Feature Extraction using MFCC
3. Emotion-wise Label Encoding
4. CNN Model Training & Evaluation
5. Model & Encoder Saving
6. Flask Web App Creation
7. Public Deployment via ngrok

✅ Resume
✅ GitHub
✅ College Submission Ready

---

---

## 📘 2 — Mount Google Drive & Extract Dataset

This step:

* Mounts Google Drive
* Extracts RAVDESS ZIP dataset
* Prepares dataset directory for processing

# ===============================

# ✅ CELL 1: Mount Drive & Extract Dataset

# ===============================

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Change this path if your ZIP is elsewhere in Drive
data_zip_path = '/content/drive/My Drive/Sasi Projects/ravdess_data.zip'


---

✅ **Yes, you can absolutely use the Kaggle Speech Emotion Dataset instead of manually uploading audio to Drive** — and it’s actually a **better, cleaner, and more reproducible approach** for your project 👏

Dataset Link:
👉 [https://www.kaggle.com/datasets/huebitsvizg/speech-emotion-dataset](https://www.kaggle.com/datasets/huebitsvizg/speech-emotion-dataset)

This means you will **REMOVE Google Drive mounting completely** and **LOAD the dataset directly from Kaggle into `/content/`**.

---

## ✅ WHAT YOU SHOULD REPLACE (Your Old Code ❌)

You will **REMOVE** your existing code block that does:

```python
from google.colab import drive
drive.mount('/content/drive')

# Change this path if your ZIP is elsewhere in Drive
data_zip_path = '/content/drive/My Drive/Sasi Projects/ravdess_data.zip'

import zipfile, os

extract_path = '/content/drive/My Drive/Sasi Projects/ravdess_data'
if not os.path.exists(extract_path):
    with zipfile.ZipFile(data_zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)

print("✅ Dataset extracted to:", extract_path)
```

and any subsequent references to `drive` paths for dataset files.

---

## ✅ NEW PROFESSIONAL KAGGLE DATASET SETUP (FINAL ✅)

### 📘 New Notebook Cell — *Download Dataset from Kaggle*

```python
# ===============================
# ✅ CELL: Install Kaggle API
# ===============================
!pip install -q kaggle
```

---

### 📘 Upload Kaggle API Key (ONE TIME STEP)

1. Go to 👉 **[https://www.kaggle.com/settings](https://www.kaggle.com/settings)**
2. Scroll to **API**
3. Click **Create New Token** — a file named `kaggle.json` will download
4. Upload it to Colab using:

```python
from google.colab import files
files.upload()
```

---

### 📘 Configure Kaggle & Download Dataset

```python
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
```

```python
# ✅ Download Speech Emotion Dataset
!kaggle datasets download -d huebitsvizg/speech-emotion-dataset
```

---

### 📘 Extract Dataset

```python
!unzip -q speech-emotion-dataset.zip
```

---

### 📘 Set Dataset Path & Load Data ✅

```python
import os
import pandas as pd

DATA_DIR = "/content/speech-emotion-dataset"

# If dataset includes a CSV or a metadata file, you can read it;
# or else walk through folders for .wav files, etc.
print("✅ Dataset Path:", DATA_DIR)

# Example (if there is a metadata CSV):
# df = pd.read_csv(os.path.join(DATA_DIR, "metadata.csv"))
# print("✅ Dataset Loaded — shape:", df.shape)
```

Or if it's purely audio files, set:

```python
DATA_PATH = "/content/speech-emotion-dataset"
```

Then you can use this `DATA_PATH` in all your later code (feature extraction, audio loading, model training, etc.).

---

## ✅ WHY THIS APPROACH IS BETTER

| Old Method                        | New Method                      |
| --------------------------------- | ------------------------------- |
| Manual ZIP upload via Drive       | ✅ Direct Kaggle download        |
| Risk of missing or outdated files | ✅ Clean structured dataset      |
| Manual extraction path handling   | ✅ Easy reproducibility in Colab |
| Drive-dependent                   | ✅ Works anywhere — shareable    |

✅ **You should now reference dataset files using locally downloaded paths**, e.g.:

```python
DATA_PATH = "/content/speech-emotion-dataset"
```

In [ ]:
import zipfile, os

extract_path = '/content/drive/My Drive/Sasi Projects/ravdess_data'
if not os.path.exists(extract_path):
    with zipfile.ZipFile(data_zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)

print("✅ Dataset extracted to:", extract_path)



---

## 📘 3 — Import Required Libraries

This step imports:

* Audio Processing Libraries
* Machine Learning Libraries
* Deep Learning Frameworks

# ===============================

# ✅ CELL 2: Import Libraries

# ===============================



In [ ]:
import os
import librosa
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from sklearn.metrics import classification_report, accuracy_score, f1_score



---

## 📘 4 — Test Single Audio File & MFCC Extraction

This step:

* Loads a test `.wav` file
* Extracts MFCC features
* Validates feature shape

# ===============================

# ✅ CELL 3: Test Feature Extraction

# ===============================



In [ ]:
def extract_features(file_path, max_pad_len=174):
    try:
        audio, sample_rate = librosa.load(file_path, res_type='kaiser_fast')
        mfccs = librosa.feature.mfcc(y=audio, sr=sample_rate, n_mfcc=40)
        pad_width = max_pad_len - mfccs.shape[1]
        mfccs = np.pad(mfccs, pad_width=((0, 0), (0, pad_width)), mode='constant')
        return mfccs
    except Exception as e:
        print("Error encountered while parsing file: ", file_path)
        return None



---

## 📘 5 — Audio File Analysis & Playback

This step:

* Reads audio metadata
* Displays duration
* Plays sample audio in Colab

# ===============================

# ✅ CELL 4: Audio Inspection & Playback

# ===============================


---



In [ ]:
import librosa
import numpy as np
import soundfile as sf

# Path to your single .wav file
file_path = "/content/drive/MyDrive/Sasi Projects/ravdess_data/Actor_01/03-01-01-01-01-01-01.wav"

def extract_features(file_path, max_pad_len=174):
    try:
        # Read audio file using soundfile (more reliable for RAVDESS)
        audio, sample_rate = sf.read(file_path, dtype='float32')

        # Convert to mono if stereo
        if len(audio.shape) > 1:
            audio = np.mean(audio, axis=1)

        # Compute MFCCs
        mfccs = librosa.feature.mfcc(y=audio, sr=sample_rate, n_mfcc=40)

        # Pad / truncate to consistent length
        pad_width = max_pad_len - mfccs.shape[1]
        if pad_width > 0:
            mfccs = np.pad(mfccs, ((0, 0), (0, pad_width)), mode='constant')
        else:
            mfccs = mfccs[:, :max_pad_len]

        print("✅ MFCC shape:", mfccs.shape)
        return mfccs

    except Exception as e:
        print("❌ Error encountered while parsing file:", file_path)
        print("Error message:", e)
        return None

# Run test
features = extract_features(file_path)
if features is not None:
    print("Feature extraction successful!")
else:
    print("Feature extraction failed.")


---

## 📘 6 — Full Dataset Feature Extraction

This step:

* Walks through all RAVDESS audio files
* Extracts MFCC features
* Maps emotion labels
* Builds feature & label arrays

# ===============================

# ✅ CELL 5: Extract MFCC From Full Dataset

# ===============================



In [ ]:
import soundfile as sf
import librosa
import numpy as np

# Path to your .wav file
file_path = "/content/drive/MyDrive/Sasi Projects/ravdess_data/Actor_01/03-01-01-01-01-01-01.wav"

# --- Option 1: Basic file info using soundfile ---
info = sf.info(file_path)
print("📁 File Info:")
print(info)

# --- Option 2: Load audio data using librosa ---
audio, sr = librosa.load(file_path, sr=None)
print("\n🎧 Audio Properties:")
print(f"Sample Rate: {sr} Hz")
print(f"Total Samples: {len(audio)}")
print(f"Duration: {len(audio)/sr:.2f} seconds")

# --- Option 3: If stereo, show shape ---
data, samplerate = sf.read(file_path)
print(f"\nAudio shape: {data.shape}")
if len(data.shape) > 1:
    print(f"Channels: {data.shape[1]} (Stereo)")
else:
    print("Channels: 1 (Mono)")

# --- Option 4: Listen to audio in Colab ---
from IPython.display import Audio
Audio(file_path)



---

## 📘 7 — Label Encoding & Train-Test Split

This step:

* Encodes emotion labels
* Converts to categorical format
* Splits dataset into Training & Testing

# ===============================

# ✅ CELL 6: Encode Labels & Split Data

# ===============================

In [ ]:
import os
import numpy as np
import librosa
from tqdm import tqdm

# Your dataset folder path (change this if different)
dataset_path = "/content/drive/MyDrive/Sasi Projects/ravdess_data"

# Emotion labels mapping from RAVDESS naming convention
emotion_labels = {
    '01': 'neutral',
    '02': 'calm',
    '03': 'happy',
    '04': 'sad',
    '05': 'angry',
    '06': 'fearful',
    '07': 'disgust',
    '08': 'surprised'
}

# Feature extraction function
def extract_features(file_path):
    try:
        y, sr = librosa.load(file_path, sr=None, duration=3, offset=0.5)
        mfccs = np.mean(librosa.feature.mfcc(y=y, sr=sr, n_mfcc=40).T, axis=0)
        return mfccs
    except Exception as e:
        print(f"❌ Error processing {file_path}: {e}")
        return None

# Collect features and labels
features, labels = [], []

print("🎧 Extracting features from all RAVDESS .wav files...\n")

for root, _, files in os.walk(dataset_path):
    for file in tqdm(files):
        if file.endswith(".wav"):
            emotion_code = file.split("-")[2]
            emotion = emotion_labels.get(emotion_code)
            if emotion:
                file_path = os.path.join(root, file)
                feature = extract_features(file_path)
                if feature is not None:
                    features.append(feature)
                    labels.append(emotion)

# Convert to NumPy arrays
X = np.array(features)
y = np.array(labels)

print(f"\n✅ Feature extraction completed successfully!")
print(f"Total audio files processed: {len(X)}")
print(f"Unique emotions found: {set(y)}")




---

## 📘 8 — CNN Model Architecture Creation

This step:

* Builds a CNN Model
* Applies Convolution, Pooling & Dropout
* Outputs Multi-Class Emotion Predictions

# ===============================

# ✅ CELL 7: Build CNN Model

# ===============================


In [ ]:
X = np.array(features)
y = np.array(labels)

# Encode labels
lb = LabelEncoder()
y_encoded = to_categorical(lb.fit_transform(y))

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

# CNN expects 4D input
X_train = X_train[..., np.newaxis]
X_test = X_test[..., np.newaxis]

print("Training shape:", X_train.shape, "Testing shape:", X_test.shape)

In [ ]:
model = Sequential([
    Conv2D(32, (3,1), activation='relu', input_shape=(40,1,1)),
    MaxPooling2D((2,1)),
    Dropout(0.3),

    Conv2D(64, (3,1), activation='relu'),
    MaxPooling2D((2,1)),
    Dropout(0.3),

    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(y_encoded.shape[1], activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()


---

## 📘 9 — Model Training

This step:

* Trains CNN on MFCC features
* Uses validation data
* Displays training performance

# ===============================

# ✅ CELL 8: Train CNN Model

# ===============================



In [ ]:
history = model.fit(
    X_train, y_train,
    epochs=30,
    batch_size=32,
    validation_data=(X_test, y_test),
    verbose=1
)



---

## 📘 10 — Model Evaluation & Metrics

This step:

* Evaluates Accuracy
* Computes F1-Score
* Prints Classification Report

# ===============================

# ✅ CELL 9: Model Evaluation

# ===============================


In [ ]:
y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true = np.argmax(y_test, axis=1)

print("✅ Accuracy:", accuracy_score(y_true, y_pred_classes))
print("✅ F1 Score:", f1_score(y_true, y_pred_classes, average='weighted'))
print(classification_report(y_true, y_pred_classes, target_names=lb.classes_))



---

## 📘 11 — Save Trained Model & Label Encoder

This step stores:

* Trained CNN Model
* Label Encoder
* Used for Flask Inference

# ===============================

# ✅ CELL 10: Save Model & Encoder

# ===============================



In [ ]:
model.save('ravdess_cnn_model.h5')

import pickle
with open('label_encoder.pkl', 'wb') as f:
    pickle.dump(lb, f)

print("✅ Model and label encoder saved successfully.")



---

## 📘 12 — Install Flask, Ngrok & Audio Dependencies

This step installs:

* Flask
* pyngrok
* librosa
* soundfile
* resampy

# ===============================

# ✅ CELL 11: Install Deployment Libraries

# ===============================

---

## 📘 13 — Create Flask Web Application

This step builds:

* Audio Upload API
* MFCC Extraction Pipeline
* Emotion Prediction API
* HTML UI with Audio Playback

# ===============================

# ✅  Create Flask App

# ===============================

---

## 📘 14 — Ngrok Public Deployment

This step:

* Authenticates ngrok
* Creates public URL
* Deploys Flask App

# ===============================

# ✅  Run Flask & Ngrok

# ===============================
---

## 🌐 Ngrok Setup (Public Deployment)

Ngrok provides a **secure public HTTPS link** to your locally running Flask application.

🔐 **For security reasons, your ngrok token should NOT be shared publicly.**

### ✅ To Use Ngrok, Follow These Steps:

### 📌 Step 1 — Get Your Auth Token

Go to this link and copy your personal token:
👉 **[https://dashboard.ngrok.com/get-started/your-authtoken](https://dashboard.ngrok.com/get-started/your-authtoken)**

---

### 📌 Step 2 — Add Token Inside Notebook

Paste your token in the following line:

```python
#from pyngrok import ngrok, conf

#conf.get_default().auth_token = "YOUR_NGROK_TOKEN_HERE"
```

---

### 📌 Step 3 — Start Ngrok Tunnel

```python
#public_url = ngrok.connect(8000)
#print("🌍 Public URL:", public_url)
```

✅ After running this, a **shareable public link** will appear here.
You can open it in your browser and access your Flask app from **anywhere in the world** 🌎

---

### ✅ Summary

✔ Secure HTTPS URL

✔ No port forwarding required

✔ Works on Google Colab

✔ Perfect for project demos, reviews, and viva

---


In [ ]:
!pip install resampy==0.4.3


In [ ]:
!pip install flask pyngrok librosa soundfile resampy==0.4.3 tensorflow

In [ ]:
from flask import Flask, request, render_template_string
from pyngrok import ngrok
import librosa, numpy as np, tensorflow as tf, pickle, os

# --- Flask setup ---
app = Flask(__name__)

# --- Load trained model + label encoder ---
model = tf.keras.models.load_model('ravdess_cnn_model.h5')
lb = pickle.load(open('label_encoder.pkl', 'rb'))

# --- Beautiful HTML template ---
HTML = """
<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <title>🎧 Speech Emotion Recognition</title>
  <style>
    body {
      font-family: 'Poppins', sans-serif;
      background: linear-gradient(135deg, #0f2027, #203a43, #2c5364);
      color: #fff;
      text-align: center;
      height: 100vh;
      margin: 0;
      display: flex;
      flex-direction: column;
      justify-content: center;
    }
    h1 {
      font-size: 2.5rem;
      margin-bottom: 0.5rem;
      color: #00e0ff;
    }
    .card {
      background: rgba(255, 255, 255, 0.1);
      border-radius: 20px;
      padding: 40px;
      width: 420px;
      margin: auto;
      box-shadow: 0 0 20px rgba(0, 0, 0, 0.3);
    }
    input[type="file"] {
      margin: 20px 0;
      padding: 10px;
      border-radius: 10px;
      background: #fff;
      color: #000;
      cursor: pointer;
    }
    input[type="submit"] {
      background: #00e0ff;
      color: #000;
      border: none;
      border-radius: 10px;
      padding: 10px 25px;
      font-size: 1rem;
      cursor: pointer;
      transition: 0.3s;
    }
    input[type="submit"]:hover {
      background: #03a9f4;
      color: #fff;
    }
    audio {
      margin-top: 15px;
      width: 100%;
    }
    .prediction {
      margin-top: 20px;
      font-size: 1.4rem;
      font-weight: 600;
      color: #00e0ff;
    }
  </style>
</head>
<body>
  <div class="card">
    <h1>🎤 Speech Emotion Recognition</h1>
    <form method="POST" enctype="multipart/form-data">
      <input type="file" name="audio" accept=".wav" required>
      <br>
      <input type="submit" value="Predict Emotion">
    </form>
    {% if prediction %}
      <audio controls>
        <source src="{{ url_for('static', filename='input.wav') }}" type="audio/wav">
        Your browser does not support the audio element.
      </audio>
      <div class="prediction">Predicted Emotion: {{ prediction }}</div>
    {% endif %}
  </div>
</body>
</html>
"""

# --- Feature extraction ---
def extract_mfcc(file_path):
    audio, sr = librosa.load(file_path, sr=None, res_type='kaiser_fast')
    mfccs = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=40)
    mfccs = np.mean(mfccs, axis=1)
    mfccs = mfccs.reshape(40, 1, 1)
    return mfccs

# --- Flask route ---
@app.route('/', methods=['GET', 'POST'])
def index():
    prediction = None
    if request.method == 'POST':
        try:
            f = request.files['audio']
            path = os.path.join('static', 'input.wav')
            os.makedirs('static', exist_ok=True)
            f.save(path)

            feature = extract_mfcc(path)
            feature = feature[np.newaxis, ...].astype(np.float32)
            pred = model.predict(feature)
            emotion = lb.classes_[np.argmax(pred)]
            prediction = emotion
        except Exception as e:
            prediction = f"❌ Error: {e}"
    return render_template_string(HTML, prediction=prediction)

# --- Ngrok tunnel ---
ngrok.set_auth_token("PASTE_YOUR_NGROK_TOKEN_HERE")  # your token
public_url = ngrok.connect(5000)
print("🌐 Public URL:", public_url.public_url)

app.run(port=5000, use_reloader=False)



---

## 📘 15 — Notebook Completed

# 🎉 Speech Emotion Recognition System Ready!

You can now:

1. Upload `.wav` Audio File
2. Detect Speaker Emotion using AI
3. Listen to Uploaded Audio
4. View Predicted Emotion
5. Access Public Web App via ngrok

✅ Fully Offline AI Model

✅ No External API Used

✅ Resume, GitHub & College Submission Ready

---

